# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"Dataset Name: {getattr(metadata, 'name', 'N/A')}")
print(f"Description: {getattr(metadata, 'description', 'N/A')}")
print(f"Cite as: {getattr(metadata, 'cite_as', getattr(metadata, 'citeAs', 'N/A'))}")


## 2. Data Overview
Review available record sets, fields, and their IDs.

The Croissant schema includes one or more "record sets" (tables/collections of records), each of which contains fields. We'll enumerate these by their `@id`.

In [ ]:
# List all record sets in the dataset and enumerate their fields
record_sets = getattr(metadata, 'record_sets', getattr(metadata, 'recordSet', []))

if not record_sets:
    print("No record sets detected directly in top-level metadata. Attempting alternate method to detect available record sets from the dataset object...")
    record_sets = dataset.record_sets

if not record_sets:
    print("No record sets found in this dataset. Please check the schema or data structure.")
else:
    for rs in record_sets:
        if hasattr(rs, '@id'):
            rs_id = rs['@id'] if isinstance(rs, dict) else rs.__dict__.get('@id', str(rs))
        else:
            rs_id = str(rs)  # Should be '@id' field
        print(f"Record Set: {rs_id}")
        if hasattr(rs, 'fields'):
            fields = rs.fields
        elif hasattr(rs, 'field'):
            fields = rs.field
        elif isinstance(rs, dict) and 'field' in rs:
            fields = rs['field']
        else:
            fields = []
        print("  Fields (by @id):")
        for field in fields:
            field_id = field['@id'] if isinstance(field, dict) and '@id' in field else str(field)
            field_name = field.get('name', '') if isinstance(field, dict) else ''
            print(f"    {field_id}  {('['+field_name+']' if field_name else '')}")
        print("")

# For demonstration, list at least one record set ID and some fields for further use
# If the library supports `.record_sets` property, we also use it to probe
probe_record_sets = dataset.record_sets if hasattr(dataset, 'record_sets') else []
displayed_record_set_ids = []
for rs in probe_record_sets:
    print(f"Record Set Found: {rs['@id']} - name: {rs.get('name', '')}")
    displayed_record_set_ids.append(rs['@id'])
    if 'fields' in rs and rs['fields']:
        for f in rs['fields']:
            print(f"   Field: {f['@id']} - name: {f.get('name', '')}")

# If record_sets were not found, attempt to infer the main record set id(s) from dataset.records signature


## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# For this dataset, if we cannot find the record set structure directly, we'll attempt to enumerate available ones
record_set_ids = []
if hasattr(dataset, 'record_sets') and dataset.record_sets:
    record_set_ids = [rs['@id'] for rs in dataset.record_sets]
elif hasattr(dataset, 'recordSet') and dataset.recordSet:
    record_set_ids = [rs['@id'] if isinstance(rs, dict) and '@id' in rs else str(rs) for rs in dataset.recordSet]
else:
    # Fallback: Try using known record set @id typical patterns (e.g. their URLs)
    # For most croissant schemas, default record set id = <dataset_id>#<table_name> or similar.
    # Example fallback for this dataset (manually inferred):
    record_set_ids = ['https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json#RecordSet:MassSpecResults']

print(f"Using record set IDs: {record_set_ids}")

dataframes = {}
for record_set_id in record_set_ids:
    try:
        records = list(dataset.records(record_set=record_set_id))
        dataframes[record_set_id] = pd.DataFrame(records)
        print(f"Loaded: {record_set_id} with {len(records)} records. Columns: {dataframes[record_set_id].columns.tolist()}")
    except Exception as e:
        print(f"Could not load records for {record_set_id}: {e}")

# Preview the columns and the first few rows
for rs_id, df in dataframes.items():
    print(f"\nRecord set {rs_id} columns:\n{df.columns.tolist()}")
    display(df.head())


## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section should include operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.
*In this cell, we'll pick likely numeric fields such as 'coverage (%)', 'MW', or 'Peptide_Count', and group by another key, e.g., 'ModDescription' or 'Description' depending on the fields present. All accesses are via their respective `@id`s, but we check their readable column names for demonstration.*

In [ ]:
# Choose a record set and relevant numeric/group fields by inspecting the DataFrame

if dataframes:
    main_rs_id = record_set_ids[0]
    df = dataframes[main_rs_id]

    columns_lower = [c.lower() for c in df.columns]

    # Select a numeric field - prioritize typical names in mass spec data
    preferred_numeric_fields = ['coverage', 'mw', 'peptide_count', 'normalized abundance']
    numeric_field = None
    for nb in preferred_numeric_fields:
        for col in df.columns:
            if nb in col.lower():
                numeric_field = col
                break
        if numeric_field:
            break
    if not numeric_field:
        numeric_field = df.select_dtypes(include=['float', 'int']).columns[0] if not df.select_dtypes(include=['float', 'int']).empty else df.columns[0]
    print(f"Selected numeric field: {numeric_field}")

    # Select a group field
    preferred_group_fields = ['moddescription', 'description', 'accession']
    group_field = None
    for gb in preferred_group_fields:
        for col in df.columns:
            if gb in col.lower():
                group_field = col
                break
        if group_field:
            break
    if not group_field:
        group_field = df.columns[0]
    print(f"Selected group field: {group_field}")

    threshold = 10    # Example threshold for numeric field

    # Filter records with field > threshold
    try:
        filtered_df = df[df[numeric_field].astype(float) > threshold]
        print(f"Filtered records with {numeric_field} > {threshold}:")
        display(filtered_df.head())

        # Normalize
        filtered_df = filtered_df.copy()
        filtered_df[f"{numeric_field}_normalized"] = (
            filtered_df[numeric_field].astype(float) - filtered_df[numeric_field].astype(float).mean()
        ) / filtered_df[numeric_field].astype(float).std()
        print(f"Normalized {numeric_field} for filtered records:")
        display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

        # Group by group_field if it exists
        if group_field in df.columns:
            grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
            print(f"Grouped data by {group_field} (mean values):")
            display(grouped_df.head())
    except Exception as e:
        print(f"Could not perform filtering and normalization: {e}")
else:
    print("No loaded dataframes to perform EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.
We'll create a histogram of the numeric field and a boxplot grouped by a selected categorical field.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if dataframes:
    main_rs_id = record_set_ids[0]
    df = dataframes[main_rs_id]

    # Re-select numeric_field and group_field for plotting (from previous cell logic)
    numeric_field = numeric_field if 'numeric_field' in locals() else df.columns[0]
    group_field = group_field if 'group_field' in locals() else df.columns[0]

    # Histogram
    plt.figure(figsize=(8, 4))
    df[numeric_field] = pd.to_numeric(df[numeric_field], errors='coerce')
    df[numeric_field].dropna().hist(bins=30)
    plt.xlabel(numeric_field)
    plt.ylabel('Count')
    plt.title(f'Distribution of {numeric_field}')
    plt.show()

    # Boxplot by group (show for top 10 categories to avoid overplotting)
    top_groups = df[group_field].value_counts().index[:10]
    plt.figure(figsize=(12, 5))
    sns.boxplot(x=group_field, y=numeric_field, data=df[df[group_field].isin(top_groups)])
    plt.xticks(rotation=75)
    plt.title(f'{numeric_field} grouped by {group_field} (top 10 categories)')
    plt.show()
else:
    print("No dataframes with loaded data to visualize.")

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.
- We successfully loaded the dataset using the `mlcroissant` library by referencing all record sets and fields by their `@id`.
- Preliminary exploration shows available fields such as protein coverage, molecular weight, and peptide abundance per record.
- Exploratory analysis demonstrates basic filtering, normalization, and grouping by descriptive fields for further protein expression analysis.
- Visualizations provide insight into the distribution of essential mass spectrometry protein data for further research.